In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
# -------------------------------
# FOLDERS
# -------------------------------
video_folder = r"D:\IP PROJECT FINAL\videos"
frame_folder = r"D:\IP PROJECT FINAL\ALL FRAMES"
gray_folder = r"D:\IP PROJECT FINAL\ALL GRAYSCALES"


# Denoising
denoise_base = r"D:\IP PROJECT FINAL\Denoised_frames"
# Illumination enhancement
enhanced_folder = r"D:\IP PROJECT FINAL\enhanced"
comparison_base = os.path.join(enhanced_folder, "Comparison")
comparison_base1=os.path.join(denoise_base, "Comparison")
# Motion blur / sharpening
final_folder = r"D:\IP PROJECT FINAL\enhanced_final"
sharpen_folder = os.path.join(final_folder, "sharpened")
edge_folder = os.path.join(final_folder, "edge_comparison")
# Grayscale image folders
gray_folders = [
    r"D:\IP PROJECT FINAL\Selected_gray_frames\aligetor",
    r"D:\IP PROJECT FINAL\Selected_gray_frames\corrugation",
    r"D:\IP PROJECT FINAL\Selected_gray_frames\edge_tracking",
    r"D:\IP PROJECT FINAL\Selected_gray_frames\logitudanal",
    r"D:\IP PROJECT FINAL\Selected_gray_frames\potholes"
]

os.makedirs(frame_folder, exist_ok=True)
os.makedirs(gray_folder, exist_ok=True)
os.makedirs(denoise_base, exist_ok=True)
os.makedirs(enhanced_folder, exist_ok=True)
os.makedirs(comparison_base, exist_ok=True)
os.makedirs(sharpen_folder, exist_ok=True)
os.makedirs(edge_folder, exist_ok=True)

# -------------------------------
# PARAMETERS
# -------------------------------
frame_skip = 15
noise_threshold = 40.0
blur_threshold = 100
    
                                                                #extract frames
                                                    
def extract_frames(video_folder, frame_output_folder, gray_output_folder, frame_skip=15):

    os.makedirs(frame_output_folder, exist_ok=True)
    os.makedirs(gray_output_folder, exist_ok=True)

    frame_id = 0

    for file in os.listdir(video_folder):

        if file.endswith((".mp4", ".avi", ".mov", ".mkv")):

            video_path = os.path.join(video_folder, file)
            cap = cv2.VideoCapture(video_path)

            if not cap.isOpened():
                print(f"Error loading {file}")
                continue

            print(f"Processing {file}...")

            video_frame_id = 0

            while True:
                ret, frame = cap.read()

                if not ret:
                    break

                if video_frame_id % frame_skip == 0:

                    frame = cv2.resize(frame, (1280, 720))

                    # Save color frame
                    save_path = os.path.join(frame_output_folder, f"{frame_id}.jpg")
                    cv2.imwrite(save_path, frame)

                    # Convert to grayscale
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                    save_path_gray = os.path.join(gray_output_folder, f"{frame_id}.jpg")
                    cv2.imwrite(save_path_gray, gray)

                    frame_id += 1

                video_frame_id += 1

            cap.release()

    print("All videos processed successfully!")
def denoise_image(img):
    return cv2.GaussianBlur(img, (5,5), 0)


# -----------------------------
# Function: Check Noise Level
# -----------------------------
def check_noise(img):
    std_dev = np.std(img)
    if std_dev < noise_threshold:
        return "Less Noise", std_dev
    else:
        return "Noisy", std_dev


# -----------------------------
# Function: Save Comparison
# -----------------------------
def save_denoise_comparison(original, denoised, folder_name, file, save_folder):

    hist_orig = cv2.calcHist([original], [0], None, [256], [0,256])
    hist_denoised = cv2.calcHist([denoised], [0], None, [256], [0,256])

    fig, axes = plt.subplots(2,2, figsize=(10,8))

    # Original Image
    axes[0,0].imshow(original, cmap='gray')
    axes[0,0].set_title("Original Image")
    axes[0,0].axis('off')

    # Denoised Image
    axes[0,1].imshow(denoised, cmap='gray')
    axes[0,1].set_title("Denoised Image")
    axes[0,1].axis('off')

    # Original Histogram
    axes[1,0].plot(hist_orig)
    axes[1,0].set_title("Original Histogram")
    axes[1,0].set_xlim([0,255])

    # Denoised Histogram
    axes[1,1].plot(hist_denoised, color='red')
    axes[1,1].set_title("Denoised Histogram")
    axes[1,1].set_xlim([0,255])

    plt.tight_layout()

    save_path = os.path.join(save_folder, f"{folder_name}_{file[:-4]}.png")
    plt.savefig(save_path)
    plt.show()
    plt.close()



                                                #analyze illumination






def analyze_illumination(image_path):
    """Analyze illumination: mean brightness, shadow %, overexposed %"""
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean_brightness = np.mean(gray)
    shadow_pixels = np.sum(gray < 50) / gray.size * 100
    bright_pixels = np.sum(gray > 200) / gray.size * 100
    return {'mean': mean_brightness, 'shadow%': shadow_pixels, 'overexposed%': bright_pixels, 'gray': gray}

def contrast_stretch(img_gray):
    """Contrast stretching using 2nd and 98th percentiles"""
    p2, p98 = np.percentile(img_gray, (2, 98))
    stretched = np.clip((img_gray - p2) * 255.0 / (p98 - p2), 0, 255)
    return stretched.astype(np.uint8)

def enhance_image(img, method='clahe'):
    """Enhancement methods: clahe, gamma, hist_eq"""
    if method == 'clahe':
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        l = clahe.apply(l)
        enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    elif method == 'gamma':
        gray_mean = np.mean(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))
        gamma = 1.5 if gray_mean < 100 else 0.7
        enhanced = np.power(img / 255.0, gamma) * 255.0
        enhanced = enhanced.astype(np.uint8)
    elif method == 'hist_eq':
        yuv = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)
        yuv[:,:,0] = cv2.equalizeHist(yuv[:,:,0])
        enhanced = cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)
    else:
        enhanced = img.copy()
    return enhanced

def save_comparison(original_img, enhanced_img, save_path):
    """Save comparison figure for original vs enhanced"""
    orig_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    enh_rgb = cv2.cvtColor(enhanced_img, cv2.COLOR_BGR2RGB)
    diff_rgb = cv2.absdiff(orig_rgb, enh_rgb)
    
    fig, axes = plt.subplots(2,3, figsize=(15,10))
    
    # Original and enhanced images
    axes[0,0].imshow(orig_rgb)
    axes[0,0].set_title("Original")
    axes[0,0].axis("off")
    
    axes[0,1].imshow(enh_rgb)
    axes[0,1].set_title("Enhanced")
    axes[0,1].axis("off")
    
    axes[0,2].imshow(diff_rgb)
    axes[0,2].set_title("Difference")
    axes[0,2].axis("off")
    
    # Histograms
    axes[1,0].hist(orig_rgb.ravel(), bins=256, color='blue', alpha=0.7)
    axes[1,0].set_title("Original Histogram")
    
    axes[1,1].hist(enh_rgb.ravel(), bins=256, color='green', alpha=0.7)
    axes[1,1].set_title("Enhanced Histogram")
    
    # Contrast plot (mean above threshold)
    def safe_mean(arr):
        return np.mean(arr) if arr.size > 0 else 0
    
    axes[1,2].plot(range(256), [safe_mean(orig_rgb[orig_rgb > i]) for i in range(256)], label='Original')
    axes[1,2].plot(range(256), [safe_mean(enh_rgb[enh_rgb > i]) for i in range(256)], label='Enhanced')
    axes[1,2].set_title("Contrast Comparison")
    axes[1,2].legend()
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
    plt.close(fig)

                                                                #motion blur analyse

    
def variance_of_laplacian(image):
    """Sharpness measurement"""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def sharpen_image(image):
    """Sharpening filter"""
    kernel = np.array([[0,-1,0],
                       [-1,5,-1],
                       [0,-1,0]])
    return cv2.filter2D(image, -1, kernel)


def smooth_image(image):
    """Light smoothing"""
    return cv2.GaussianBlur(image, (3,3), 0)


def edge_detection(image):
    """Canny edge detection"""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    return edges


def save_edge_comparison(orig, enhanced, save_path):
    """Save edge comparison figure"""

    edges_orig = edge_detection(orig)
    edges_enh = edge_detection(enhanced)

    fig, axes = plt.subplots(1,2, figsize=(10,5))

    axes[0].imshow(edges_orig, cmap='gray')
    axes[0].set_title("Original Edges")
    axes[0].axis("off")

    axes[1].imshow(edges_enh, cmap='gray')
    axes[1].set_title("Enhanced Edges")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
    plt.close(fig)
   
    orig_edges_count = cv2.countNonZero(edges_orig)
    enh_edges_count = cv2.countNonZero(edges_enh)

    clarity_ratio = enh_edges_count / orig_edges_count if orig_edges_count != 0 else 0

    return orig_edges_count, enh_edges_count, clarity_ratio

#denoise pipeline
    
def denoise_pipeline():

    os.makedirs(denoise_base, exist_ok=True)
    os.makedirs(comparison_base, exist_ok=True)

    for folder in gray_folders:

        folder_name = os.path.basename(folder)

        save_folder = os.path.join(denoise_base, folder_name)
        compare_folder = os.path.join(comparison_base1, folder_name)

        os.makedirs(save_folder, exist_ok=True)
        os.makedirs(compare_folder, exist_ok=True)

        print(f"\nProcessing folder: {folder_name}")

        for file in os.listdir(folder):

            if file.endswith(".jpg"):

                path = os.path.join(folder, file)
                img_gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

                if img_gray is None:
                    continue

                # 1 Noise Check
                noise_status, std = check_noise(img_gray)
                print(f"{file}: {noise_status} (StdDev={std:.2f})")

                # 2 Denoise image
                img_denoised = denoise_image(img_gray)

                # 3 Save denoised image
                save_path = os.path.join(save_folder, file)
                cv2.imwrite(save_path, img_denoised)

                # 4 Save comparison figure
                save_denoise_comparison(img_gray, img_denoised, folder_name, file, compare_folder)

    print("\nAll images processed successfully!")

#illumination pipeline    
def illumination_pipeline():

    for folder in os.listdir(denoise_base):

        folder_path = os.path.join(denoise_base, folder)

        if not os.path.isdir(folder_path):
            continue

        for file in os.listdir(folder_path):

            if not file.lower().endswith(".jpg"):
                continue

            path = os.path.join(folder_path, file)

            illum = analyze_illumination(path)

            shadow = illum['shadow%']
            over = illum['overexposed%']
            mean_b = illum['mean']

            img = cv2.imread(path)

            if shadow >= 15:
                method = 'clahe'
            elif over >= 15:
                method = 'gamma'
            else:
                method = 'hist_eq'

            enhanced = enhance_image(img, method=method)

            gray_enh = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)
            stretched = contrast_stretch(gray_enh)

            final_img = cv2.cvtColor(stretched, cv2.COLOR_GRAY2BGR)

            save_path = os.path.join(enhanced_folder, f"{folder}_{file}")
            cv2.imwrite(save_path, final_img)

            comparison_path = os.path.join(comparison_base, f"comp_{folder}_{file[:-4]}.png")

            save_comparison(img, final_img, comparison_path)

    print("✅ Illumination enhancement completed.")


#sharpening pipeline
    
def sharpening_pipeline():
    for file in os.listdir(enhanced_folder):
        if not file.lower().endswith(".jpg"):
            continue
        path = os.path.join(enhanced_folder, file)
        img = cv2.imread(path)
        sharpness = variance_of_laplacian(img)
        blurry = sharpness < blur_threshold
        if blurry:
            enhanced = sharpen_image(img)
        else:
            enhanced = img.copy()
        enhanced = smooth_image(enhanced)
        save_path = os.path.join(sharpen_folder, f"{file}")
        cv2.imwrite(save_path, enhanced)
        edge_save_path = os.path.join(edge_folder, f"edge_{file[:-4]}.png")
        orig_edges, enh_edges, clarity_ratio = save_edge_comparison(img, enhanced, edge_save_path)
        print(f"{file} | Sharpness: {sharpness:.2f} | Blurry: {blurry} | Edge clarity: {clarity_ratio:.2f}")
    print("✅ Sharpening & edge analysis completed.")


#final pipeline    
def enhancement_pipeline():
    print("\n Stage 1: Frame Extraction")
    extract_frames(video_folder, frame_folder, gray_folder, frame_skip=15)
    
    print("\n Stage 2: Denoising")
    denoise_pipeline()
    
    print("\n Stage 3: Illumination Enhancement")
    illumination_pipeline()
    
    print("\n Stage 4: Sharpening & Edge Analysis")
    sharpening_pipeline()
    
    print("\n All stages completed successfully!")

enhancement_pipeline()